In [ ]:
import pandas
import matplotlib.pyplot as plt
dataset = pandas.read_csv('https://covid.ourworldindata.org/data/owid-covid-data.csv', engine='python')
dataset

In [ ]:
from pandas.core.frame import DataFrame
ds_filtered=dataset[dataset['location']=='Peru']
ds_filtered=DataFrame(data={"new_cases_smoothed":ds_filtered['new_cases_smoothed'].values},index=ds_filtered['date'].values)
ds_filtered=ds_filtered.fillna(method='bfill').fillna(method='ffill')
ds_filtered

In [ ]:
ds_filtered.plot()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

def min_max_scale(x,min,max):
  return (x-min)/(max-min)

def min_max_inverse(x_scaled,min,max):
  return (x_scaled*(max-min))+min

def scale_array(arr,min,max):
  res=[]
  for x in arr:
    res.append(min_max_scale(x,min,max))
  return np.asanyarray(res)

def inv_scale_array(arr,min,max):
  res=[]
  for x in arr:
    res.append(min_max_inverse(x,min,max))
  return np.asanyarray(res)
max=ds_filtered.max()
min=ds_filtered.min()
print(ds_filtered.values.shape)
arr_scaled=scale_array(ds_filtered.values,min,max)
ds_scaled=pandas.DataFrame(arr_scaled)
ds_scaled

In [ ]:
import numpy as np
n_output=7
def input_reshaper(df):
  n_days_input=14
  x=[]
  y=[]
  obj_list=[]
  for i in range(n_days_input,df.shape[0]-n_output):
    input_list=[]
    output_list=[]
    for j in range(0,n_days_input):
      input_list.append(df.iloc[i-(14-j)][0])
    input_mean=np.mean(input_list)
    input_std=np.std(input_list)
    input_list.append(input_mean)
    input_list.append(input_std)
    input_array=np.array(input_list)
    for k in range(0,n_output):
      output_list.append(df.iloc[i+k][0])
    output_array=np.array(output_list)
    x.append(input_array)
    #y.append(df.iloc[i][0])
    y.append(output_array)
    #print(df.iloc[i].name)
  x_array=np.array(x)
  y_array=np.array(y)
  return x_array,y_array,obj_list

input_reshaped_x,input_reshaped_y,objs=input_reshaper(ds_scaled)
print(input_reshaped_x.shape)
print(input_reshaped_y.shape)
input_reshaped_x=input_reshaped_x.reshape(-1,16,1)
input_reshaped_y=input_reshaped_y.reshape(-1,7,1)
print(input_reshaped_x.shape)
print(input_reshaped_x)

In [ ]:

from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(input_reshaped_x,input_reshaped_y,test_size=0.3,random_state=0)

In [ ]:
import tensorflow as tf
keras=tf.keras
from keras.utils.vis_utils import plot_model

model= keras.models.Sequential()
model.add(keras.layers.LSTM(units=16,activation='tanh',recurrent_activation='sigmoid',return_sequences=True,input_shape=(16,1)))
model.add(keras.layers.LSTM(units=16,activation='tanh',recurrent_activation='sigmoid',return_sequences=False))
model.add(keras.layers.Dense(n_output,activation='linear'))
model.compile(optimizer='Nadam', loss='mse', metrics=['mse'])
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True)

In [ ]:
history=model.fit(xtrain,ytrain,epochs=100,verbose=1,batch_size=16,validation_data=(xtest,ytest))

In [ ]:
train_loss = history.history['loss']
val_loss   = history.history['val_loss']
xc         = range(100)
plt.figure()
plt.plot(xc, train_loss,label='train loss')
plt.plot(xc, val_loss,label='val loss')
plt.legend()

In [ ]:
model.save('model.h5')
# make predictions


In [ ]:
import sklearn.metrics as metrics

# make predictions
testPredict = model.predict(xtest)
testPredict=testPredict.reshape(-1,7)
ytest=ytest.reshape(-1,7)
y_unscaled=[]
for y in ytest:
    y_unscaled.append(inv_scale_array(y,min,max))
predict_unscaled=[]
for p in testPredict:
    predict_unscaled.append(inv_scale_array(p,min,max))


y_unscaled=np.array(y_unscaled)
predict_unscaled=np.array(predict_unscaled)
y_unscaled=y_unscaled.reshape(-1,7)
predict_unscaled=predict_unscaled.reshape(-1,7)

print('R2: ',metrics.r2_score(y_unscaled,predict_unscaled))
print('MAE: ',metrics.mean_absolute_error(y_unscaled,predict_unscaled))
print('MSE: ',metrics.mean_squared_error(y_unscaled,predict_unscaled))
print('RMSE: ',metrics.mean_squared_error(y_unscaled,predict_unscaled,squared=False))

In [ ]:
import requests
url = 'https://f6e4-2001-1388-303-59bd-58b6-e64c-ae48-dbb.sa.ngrok.io/model/handle/new'
filename = 'model'
formdata = {"file":(filename, open('/kaggle/working/model.h5', 'rb'), "multipart/form-data")}
#request = requests.post(url, files=formdata)